# Kayıp Fonksiyonları

Bu alıştırmada, Kayıp fonksiyonlarının `LinearRegression` modeli üzerindeki etkilerini karşılaştıracaksınız.

👇 Bu zorluk için kullanmak üzere bir CSV dosyası indirelim ve onu bir DataFrame'e dönüştürelim

In [1]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/loss_functions_dataset.csv")
data.sample(5)

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Glazing Area,Average Temperature
503,0.76,661.5,416.5,122.50,7.0,0.25,36.425
146,0.98,514.5,294.0,110.25,7.0,0.10,24.975
305,0.79,637.0,343.0,147.00,7.0,0.25,38.920
258,0.79,637.0,343.0,147.00,7.0,0.10,39.955
319,0.71,710.5,269.5,220.50,3.5,0.25,14.220


🎯 Göreviniz, tasarımına göre bir seranın içindeki ortalama sıcaklığı tahmin etmektir. Sıcaklık tahminleriniz, her bir bitki için iklim ihtiyaçlarına göre uygun sera tasarımını seçmenize yardımcı olacaktır.

🌿 Bitkilerin küçük sıcaklık değişimlerini kaldırabildiğini, ancak sıcaklık değişimleri arttıkça katlanarak daha duyarlı hale geldiğini biliyorsunuz.

## 1. Teori

❓ Teorik olarak, bitkileri öldürme riskini sınırlamak için modelinizi hangi Kayıp fonksiyonu üzerinde eğitirsiniz?

<details>
<summary> 🆘 Cevap </summary>
    
Teorik olarak, Ortalama Kare Hata (MSE) Kayıp fonksiyonunu kullanırsınız. Bu, aykırı tahminleri cezalandırır ve modelinizin büyük hatalar yapmasını engeller. Bu, daha küçük sıcaklık değişimleri ve bitkiler için daha düşük risk sağlayacaktır.

</details>

Teorik olarak Ortalama Kare Hata (MSE) kullanmalıyım. Çünkü bitkiler sıcaklık değişimleri arttıkça katlanarak daha duyarlı hale geliyor. MSE, hataların karesini aldığı için büyük hataları çok daha ağır cezalandırır ve modelin büyük sapmalar yapmasını engelleyerek bitkileri korur.

## 2. Uygulama

### 2.1 Ön İşleme

❓ Özellikleri standartlaştırın

In [4]:
from sklearn.preprocessing import StandardScaler

X = data.drop(columns=['Average Temperature'])
y = data['Average Temperature']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### 2.2 Modelleme

Bu bölümde, farklı Kayıp fonksiyonları üzerinde optimize edilmiş modelleri değerlendirerek teoriyi doğrulayacaksınız.

### En Küçük Kareler (MSE) Kaybı

❓ **En Küçük Kareler Kaybı** (MSE) üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru ve bunu `r2` değişkeninde kaydedin
- Tüm katlarınızın °C cinsinden en büyük tek tahmin hatasını hesaplayın ve `max_error_celsius` değişkeninde kaydedin

(İpucu: `max_error` sklearn'de kabul edilen bir puanlama metriğidir)

In [5]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import cross_validate


model_mse = SGDRegressor(loss='squared_error', random_state=42)

cv_results_mse = cross_validate(model_mse, X_scaled, y, cv=10, scoring=['r2', 'max_error'])

r2 = cv_results_mse['test_r2'].mean()
max_error_celsius = abs(cv_results_mse['test_max_error']).max()

print(f"MSE - Ortalama R2: {r2}")
print(f"MSE - En Büyük Hata (°C): {max_error_celsius}")

MSE - Ortalama R2: 0.8982272875896993
MSE - En Büyük Hata (°C): 9.787281346962047


### Ortalama Mutlak Hata (MAE) Kaybı

Peki modelimizi MAE üzerinde optimize edersek ne olur?

❓ **MAE** Kaybı üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

<details>
<summary>💡 İpuçları</summary>

- MAE kaybı `SGDRegressor`'da doğrudan belirtilemez. Doğru parametreleri ayarlayarak tasarlanması gerekir

</details>

❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru, bunu `r2_mae`'de saklayın
- Tüm katlarınızın en büyük tek tahmin hatasını, bunu `max_error_mae`'de saklayın

In [6]:
# MAE için özel parametreler
model_mae = SGDRegressor(loss='epsilon_insensitive', epsilon=0, random_state=42)

cv_results_mae = cross_validate(model_mae, X_scaled, y, cv=10, scoring=['r2', 'max_error'])

r2_mae = cv_results_mae['test_r2'].mean()
max_error_mae = abs(cv_results_mae['test_max_error']).max()

print(f"MAE - Ortalama R2: {r2_mae}")
print(f"MAE - En Büyük Hata (°C): {max_error_mae}")

MAE - Ortalama R2: 0.8763275370982454
MAE - En Büyük Hata (°C): 11.210013223184902


## 3. Sonuç

❓ Değerlendirdiğiniz modellerden hangisi göreviniz için en uygun görünüyor?

<details>
<summary> 🆘Cevap </summary>
    
İki model arasında ortalama çapraz doğrulanmış r2 skorları yaklaşık olarak benzer olmasına rağmen, MAE üzerinde optimize edilen modelin zaman zaman daha büyük hatalar yapma şansı daha fazladır, bu da bitkileri öldürme riskini artırır!
    
</details>

Sera sıcaklığı tahmini göreviniz için en uygun olan model, Ortalama Kare Hata (MSE) kaybı üzerine optimize edilmiş olan modeldir.Bu seçim şu nedenlere dayanmaktadır:Büyük Hataların Cezalandırılması: Bitkiler küçük sıcaklık değişimlerine tolerans gösterebilirken, değişimler arttıkça bu duruma katlanarak daha duyarlı hale gelmektedirler.Risk Yönetimi: MSE, hataların karesini aldığı için büyük sapmaları MAE'ye (Ortalama Mutlak Hata) kıyasla çok daha ağır cezalandırır.Tahmin Tutarlılığı: Her iki modelin genel başarı skorları ($R^2$) birbirine yakın olsa bile, MAE ile eğitilen bir modelin zaman zaman çok daha büyük hatalar yapma riski daha yüksektir.Bitki Sağlığı: MSE kullanımı, uç değerlerdeki (aykırı) tahmin hatalarını sınırlayarak bitkilerin ölme riskini minimuma indirir ve daha güvenli bir iklimlendirme sağlar.;

# 🏁 Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [7]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'loss_functions',
    r2 = r2,
    r2_mae = r2_mae,
    max_error = max_error_celsius,
    max_error_mae = max_error_mae
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/bariscan/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/bariscan/S16D4-S-loss-functions/tests
plugins: typeguard-4.4.2, anyio-4.8.0
collecting ... collected 3 items

test_loss_functions.py::TestLossFunctions::test_max_error_order PASSED   [ 33%]
test_loss_functions.py::TestLossFunctions::test_r2 PASSED                [ 66%]
test_loss_functions.py::TestLossFunctions::test_r2_mae PASSED            [100%]

============================== 3 passed in 0.22s ===============================


💯 You can commit your code:

git add tests/loss_functions.pickle

git commit -m 'Completed loss_functions step'

git push origin master

